# CoALA + Planning Design Pattern
## Customer Support Agent

---

### What This Notebook Focuses On

**Fixed (CoALA structure):** Parse -> Parallel Retrieval -> Working Memory -> Decision -> Act -> Learn

**Variable (Planning pattern):** Working Memory calls the LLM **ONCE** to generate a complete plan upfront. The plan is then executed step-by-step with **zero further LLM calls**.

### The key question Planning answers:
> *"What is the full sequence of actions I need to take?"* -- decided once, before any action.

### Memory: Pinecone `coala-memory` (seeded in Notebook_5)
Semantic and episodic memory retrieved from Pinecone inform the plan generation.

## CoALA Control Flow -- Planning Variant

```
USER MESSAGE
    |
    v
[PARSE]                 -- extract intent, order_id, sentiment
    |
    v
[PARALLEL RETRIEVAL]    -- Pinecone semantic + episodic
    |
    v
[PLANNING PHASE]        -- ONE LLM call -> full JSON plan
    |
    v
[EXECUTION PHASE]       -- loop through plan, call tools directly
    |                      ZERO further LLM calls
    v
[LEARNING PHASE]        -- write facts + episode back to Pinecone
```

**Contrast with ReAct (Notebook_7):** In ReAct, the LLM is called again after every tool result.
In Planning, the LLM commits to the full plan upfront and is never consulted again during execution.

In [ ]:
import os, re, json, time
import pandas as pd
from typing import Optional
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)
embedder = SentenceTransformer("paraphrase-MiniLM-L6-v2")
orders_df = pd.read_csv("order.csv")

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("coala-memory")        # same index seeded in Notebook_5
SEMANTIC_NS = "coala_semantic"
EPISODIC_NS = "coala_episodic"

print(f"Loaded {len(orders_df)} orders")
print(f"Connected to Pinecone index: coala-memory")
print(orders_df.head(3))

## Tools and Memory Helpers

Same `@tool` functions and Pinecone helpers as Notebook_5. Shared across NB6, NB7, NB8.

In [ ]:
@tool
def fetch_order(order_id: int) -> str:
    """Fetch full order details from the database given an order ID."""
    result = orders_df[orders_df["order_id"] == order_id]
    if result.empty:
        return f"No order found with ID {order_id}."
    return json.dumps(result.iloc[0].to_dict(), default=str)

@tool
def check_shipping_status(order_id: int) -> str:
    """Get a human-readable shipping status message for an order."""
    result = orders_df[orders_df["order_id"] == order_id]
    if result.empty:
        return f"No order found with ID {order_id}."
    status = result.iloc[0]["status"]
    return {"Delivered": "Your order has been delivered.",
            "Shipped": "Your order is currently on the way.",
            "Processing": "Your order is still being prepared and has not shipped yet.",
            "Cancelled": "Your order has been cancelled."}.get(status, "Status unknown.")

@tool
def offer_compensation(order_id: int) -> str:
    """Apply a 10% discount to the customer's account as compensation for an order issue."""
    result = orders_df[orders_df["order_id"] == order_id]
    name = result.iloc[0]["user_name"] if not result.empty else "the customer"
    return f"10% discount successfully applied to {name}'s account."

@tool
def provide_order_info(order_id: int) -> str:
    """Provide detailed order information: product name, status, and order date."""
    result = orders_df[orders_df["order_id"] == order_id]
    if result.empty:
        return f"No order found with ID {order_id}."
    r = result.iloc[0]
    return f"Order #{r['order_id']} -- {r['product_name']}, Status: {r['status']}, Placed on: {r['date']}."

@tool
def escalate_to_human(order_id: Optional[int] = None) -> str:
    """Escalate a complex or unresolved issue to the human support team. order_id is optional."""
    suffix = f" for order {order_id}" if order_id else ""
    return f"The issue{suffix} has been escalated. A representative will contact you within 24 hours."

action_tools = [fetch_order, check_shipping_status, offer_compensation, provide_order_info, escalate_to_human]
tool_registry = {t.name: t for t in action_tools}

In [ ]:
def retrieve_semantic(query: str, k: int = 3) -> list:
    qv = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    res = index.query(vector=qv, top_k=k, include_metadata=True, namespace=SEMANTIC_NS)
    hits = [m["metadata"]["text"] for m in res.get("matches", []) if m["score"] > 0.25]
    print(f"  [SemanticMemory] {len(hits)} facts retrieved")
    return hits

def retrieve_episodic(query: str, k: int = 2) -> list:
    qv = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    res = index.query(vector=qv, top_k=k, include_metadata=True, namespace=EPISODIC_NS)
    hits = [m["metadata"] for m in res.get("matches", []) if m["score"] > 0.25]
    print(f"  [EpisodicMemory] {len(hits)} past episodes retrieved")
    return hits

def learn_semantic(fact: str):
    vec = embedder.encode([fact], normalize_embeddings=True)[0].tolist()
    vid = f"fact_{int(time.time())}_{abs(hash(fact)) % 100000}"
    index.upsert(vectors=[(vid, vec, {"text": fact, "ts": int(time.time())})], namespace=SEMANTIC_NS)
    print(f"  [SemanticMemory.learn] {fact[:70]}")

def learn_episodic(episode: dict):
    summary = episode.get("summary", str(episode))
    vec = embedder.encode([summary], normalize_embeddings=True)[0].tolist()
    vid = f"ep_{int(time.time())}_{abs(hash(summary)) % 100000}"
    meta = {"summary": summary, "intent": episode.get("intent", ""),
            "outcome": episode.get("outcome", ""), "ts": int(time.time())}
    index.upsert(vectors=[(vid, vec, meta)], namespace=EPISODIC_NS)
    print(f"  [EpisodicMemory.learn] {summary[:70]}")

def parse_observation(message: str) -> dict:
    msg = message.lower()
    match = re.search(r'\b(50\d{2})\b', msg)
    order_id = int(match.group(1)) if match else None
    if any(w in msg for w in ["delay", "late", "not arrived", "not received", "not shipped"]):
        intent = "order_delay"
    elif any(w in msg for w in ["cancel", "cancellation", "cancelled"]):
        intent = "order_cancellation"
    elif any(w in msg for w in ["where", "status", "track", "when", "update", "details"]):
        intent = "order_status"
    else:
        intent = "general_enquiry"
    sentiment = "negative" if any(w in msg for w in ["unhappy", "angry", "frustrated", "unacceptable", "furious"]) else "neutral"
    print(f"  [PARSE] intent={intent}, order_id={order_id}, sentiment={sentiment}")
    return {"raw": message, "intent": intent, "order_id": order_id, "sentiment": sentiment}

## Planning Working Memory -- One LLM Call

The LLM receives the message plus all retrieved memory context, and returns a **complete JSON plan**.
This is the **only** LLM call in the entire interaction. No further reasoning happens during execution.

In [ ]:
def planning_working_memory(message: str, parsed: dict, semantic_ctx: list, episodic_ctx: list) -> list:
    """ONE LLM call -- returns the full execution plan as a list of dicts."""
    semantic_block = "\n".join(f"- {f}" for f in semantic_ctx) if semantic_ctx else "None"
    episodic_block = "\n".join(e.get("summary", "") for e in episodic_ctx) if episodic_ctx else "None"
    order_id = parsed.get("order_id")

    prompt = f"""You are a customer support planner.

Semantic memory (relevant facts):
{semantic_block}

Past episodes (similar resolutions):
{episodic_block}

Customer message: \"{message}\"
Extracted order ID: {order_id if order_id else 'NOT PROVIDED'}

Generate a COMPLETE execution plan as a JSON array of steps.
Each step: {{"tool": "<tool_name>", "args": {{"order_id": <integer or null>}}}}

Available tools:
- fetch_order: get full order details (requires order_id)
- check_shipping_status: get delivery status message (requires order_id)
- offer_compensation: apply 10% discount (requires order_id)
- provide_order_info: get product/date info (requires order_id)
- escalate_to_human: escalate to support (order_id optional)

Rules:
- Start with fetch_order if order_id is available
- Use the exact order_id integer from the message
- Choose action steps based on the customer need

Return ONLY a valid JSON array. No markdown. No explanation."""

    raw = llm.invoke(prompt).content.strip()
    # Strip markdown code fences if LLM adds them
    raw = re.sub(r"^```[a-z]*\n?", "", raw).rstrip("`").strip()
    return json.loads(raw)


def execute_plan(plan: list) -> list:
    """Execute every step in the plan -- no LLM involved."""
    results = []
    for i, step in enumerate(plan):
        tool_name = step.get("tool", "")
        args = {k: v for k, v in step.get("args", {}).items() if v is not None}
        print(f"  Step {i+1}/{len(plan)}: {tool_name}({args})")
        if tool_name in tool_registry:
            result = tool_registry[tool_name].invoke(args)
        else:
            result = f"Unknown tool: {tool_name}"
        print(f"  -> {str(result)[:90]}")
        results.append({"step": step, "result": result})
    return results


def compose_response(results: list) -> str:
    """Build final response from tool results -- no LLM call."""
    parts = []
    for r in results:
        tool_name = r["step"].get("tool", "")
        result = r["result"]
        if tool_name == "fetch_order":
            try:
                rec = json.loads(result)
                parts.append(f"Hi {rec['user_name']}! Regarding your {rec['product_name']} (Order #{rec['order_id']}):")
            except Exception:
                parts.append(result)
        elif tool_name in ["check_shipping_status", "offer_compensation", "provide_order_info", "escalate_to_human"]:
            parts.append(result)
    return " ".join(parts) if parts else "Your request has been processed."

## CoALA Planning Agent

In [ ]:
class CoALAPlanningAgent:

    def handle(self, message: str) -> str:
        print(f"\nUSER: {message}")
        print("="*60)

        # 1. PARSE
        print("\n[PARSE]")
        parsed = parse_observation(message)

        # 2. PARALLEL RETRIEVAL from Pinecone
        print("\n[PARALLEL RETRIEVAL -- Pinecone coala-memory]")
        semantic_ctx = retrieve_semantic(message)
        episodic_ctx = retrieve_episodic(message)

        # 3. PLANNING PHASE -- single LLM call
        print("\n[PLANNING PHASE] -- Calling LLM ONCE to generate the full plan")
        print("  This is the ONLY LLM call in this entire interaction.")
        plan = planning_working_memory(message, parsed, semantic_ctx, episodic_ctx)
        print(f"\n  Plan ({len(plan)} steps):")
        for i, step in enumerate(plan):
            print(f"  {i+1}. {step}")

        # 4. EXECUTION PHASE -- no LLM
        print(f"\n[EXECUTION PHASE] -- Running {len(plan)} steps -- ZERO further LLM calls")
        results = execute_plan(plan)

        # 5. COMPOSE RESPONSE -- no LLM
        response = compose_response(results)

        # 6. LEARNING PHASE -- write back to Pinecone
        print("\n[LEARNING PHASE]")
        learn_semantic(f"Planning pattern resolved {parsed['intent']}: used {len(plan)}-step plan")
        learn_episodic({
            "summary": f"{message[:60]} -> {len(plan)}-step plan, intent={parsed['intent']}",
            "intent": parsed["intent"],
            "outcome": "resolved",
        })

        print(f"\nFINAL RESPONSE:\n{response}")
        return response

In [ ]:
agent = CoALAPlanningAgent()

# Order 5003 is in 'Processing' status -- plan should: fetch -> check status -> offer compensation
agent.handle("My order 5003 has been delayed by 3 days and I am unhappy. I want compensation.")

In [ ]:
# Second interaction -- episodic memory from above is now in Pinecone
# Notice: if you run this in the same session, the episodic retrieval may find the previous episode
agent.handle("I need an update on order 5020 please.")

## Key Observations -- Planning Pattern

**What to notice in the output:**
- ONE LLM call in the Planning Phase
- The plan is printed as a JSON list before any tool runs
- Execution phase runs each step sequentially -- no LLM involvement
- Learning phase writes back to the same Pinecone index (available to NB7, NB8)

**Limitation:** The plan is fixed at generation time.
If order 5003 turns out to be Cancelled (not delayed), the plan still tries to offer compensation.
The agent has no way to adapt -- it committed upfront.
See Notebook_7 (ReAct) to see how this limitation is addressed.

---

| Aspect | Planning (this notebook) | ReAct (NB7) | Tool Use (NB8) |
|---|---|---|---|
| LLM calls per interaction | 1 (upfront) | N (one per step) | N (one per tool selection) |
| Can adapt mid-execution | No -- plan is fixed | Yes | Yes |
| Execution by | Code (plan runner) | LangChain executor | LangChain executor |
| Best for | Known, stable workflows | Uncertain, exploratory tasks | Any grounded real-data task |